In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
import torch
import warnings
import os

warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

data_path = r"C:\Users\rajeshkumar.t\Desktop\ML\PL_Outputs.xlsx"
df = pd.read_excel(data_path)

df = df[['Date', 'case_id', 'Description']].copy()
df['clean_text'] = df['Description'].fillna('').str.lower().str.replace('undefined - undefined -', '', regex=True).str.strip()
unique_texts = df['clean_text'].unique().tolist()

choices = [
    'Delay in Service - Out of SPD', 'Claim Registrations', 'Fake closure', 
    'Fake status', 'Warranty Related', 'To Check Status Of The Call', 'Delay in service'
]

print(f"Processing {len(unique_texts)} unique rows on {device.upper()}...")

# 4. Encoding
label_embeddings = model.encode(choices, convert_to_tensor=True, device=device)

unique_embeddings = model.encode(
    unique_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_tensor=True,
    device=device    
)

# 5. Fixed Similarity Calculation (Changed cos.sim to cos_sim and variable name)
cosine_scores = util.cos_sim(unique_embeddings, label_embeddings)

# 6. Extract Scores (Fixed variable names)
max_scores, best_match_indices = torch.max(cosine_scores, dim=1)
max_scores = max_scores.cpu().numpy()
best_match_indices = best_match_indices.cpu().numpy()

# 7. Map Results (Fixed typos in map names)
threshold = 0.35
results_map = {}
for i, text in enumerate(unique_texts):
    if max_scores[i] > threshold:
        results_map[text] = choices[best_match_indices[i]]
    else:
        results_map[text] = 'Miscellaneous'

# Apply the mapping to the main dataframe
df['Final_Type'] = df['clean_text'].map(results_map)

# 8. Summary (Fixed ro_string to to_string)
summary = df['Final_Type'].value_counts().reset_index()
summary.columns = ['Category', 'count']
summary['percentage'] = (summary['count'] / len(df) * 100).round(2)

print("\n" + "-"*30)
print(summary.to_string(index=False))
print("-" * 30)

# 9. Export (Fixed double brackets and pring typo)
output_columns = ['Date', 'case_id', 'Description', 'Final_Type']
output_path = r"C:\Users\rajeshkumar.t\Desktop\Test_mining.xlsx"

df[output_columns].to_excel(output_path, index=False)
print(f"\nSuccessfully updated the data: {output_path}")